# Computational Theory Problems

This notebook works through a series of problems based on the Secure Hash Standard published by NIST as FIPS PUB 180-4. 

## Problem 1: Representing SHA-256 Data

SHA-256 works on fixed-size binary data. Before implementing the algorithm, this section chooses a Python representation for each kind of data it uses, based on the definitions in FIPS 180-4.

The choice matters because Python's built-in `int` has no fixed size. SHA-256 depends on values that are exactly 32 bits wide and that wrap around when they overflow ([FIPS 180-4, Section 3.2](https://doi.org/10.6028/NIST.FIPS.180-4)). The wrong representation produces a wrong hash without raising any error.

For each kind of data, this section:

- states what the standard requires
- explains why the chosen representation is appropiate
- demonstrates the chosen representation with an example




| Data | Size | Representation |
| --- | --- | --- |
| Word | 32 bits | `np.uint32` |


Each choice is explained and demonstrated below.

### Imports

[NumPy](https://numpy.org/doc/stable/user/basics.types.html) provides fixed-width integer types such as `uint32`. The standard library module `sys` reports the machine's native byte order.

In [4]:
# Standard library: reports this machine's native byte order.
import sys

# NumPy: provides fixed-width unsigned integers such as uint32.
import numpy as np

### 1. 32-bit Words

FIPS 180-4 ([Section 2.1](https://doi.org/10.6028/NIST.FIPS.180-4)) defines a *word* as a group of $w$ bits, with $w = 32$ for SHA-256. Every SHA-256 operation works on words.

**Choice:** NumPy's `np.uint32`, an unsigned (non-negative) integer that is always exactly 32 bits wide.

**Why it is appropriate:**

- It can't hold more than 32 bits which matches the defintion of a word
- Addition wraps around after $2^{32} - 1$, as required by Section 3.2
- A lift shift removes any bit moved after the 32nd position

Each of these properties is checked below.


In [5]:
# Create a single 32-bit word holding the largest possible value.
word = np.uint32(0xFFFFFFFF)

print(word)
print(type(word))
print(hex(word))

4294967295
<class 'numpy.uint32'>
0xffffffff


**Exactly 32 bits.** A 32-bit unsigned integer ranges from $0$ to $2^{32} - 1$. Instead of entering these limits by hand, [`np.iinfo`](https://numpy.org/doc/stable/reference/generated/numpy.iinfo.html) reads them directly from NumPy.

In [6]:
# Show the smallest and largest values a uint32 can hold.
limits = np.iinfo(np.uint32)

print(limits.min, limits.max)
print(hex(limits.max))

0 4294967295
0xffffffff


**Addition wraps modulo $2^{32}$.** 

Adding 1 to the largest `uint32` value gives 0. The same sum with a Python `int` is included for contrast: it continues to 4294967296, which is not a valid 32-bit word.

An array is used here rather than a single value. Adding two single `np.uint32` values also gives 0, but NumPy prints an overflow warning as well. An array wraps without the warning.

In [7]:
# A uint32 array wraps round to 0 when it overflows.
words = np.array([0xFFFFFFFF], dtype=np.uint32)
print(words + np.uint32(1))

# A Python int keeps growing past 32 bits.
print(0xFFFFFFFF + 1)

[0]
4294967296


**Shifting stays within 32 bits.** 

The value `0x80000001` has its highest and lowest bits set. Shifting it left by one moves the highest bit outside the 32-bit range. A `uint32` removes that bit and gives `0x2`, while a Python `int` keeps it and becomes 33 bits long.

In [8]:
# A value with its highest and lowest bits set.
x = np.uint32(0x80000001)

# Shift left by one: the highest bit falls off a uint32.
print(hex(x << 1))

# The same shift on a Python int keeps the extra bit.
print(hex(0x80000001 << 1))

0x2
0x100000002
